# Setup: Prerequisites for Korean LLM Evaluation

This notebook configures the required RBAC permissions and secrets to run `LMEvalJob` against an OAuth-protected KServe InferenceService on OpenShift AI.

## What is LMEvalJob?

LMEvalJob is a Custom Resource provided by the TrustyAI Operator that runs [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness) as a Kubernetes Job. It handles dataset download, model inference, and result collection.

## Prerequisites

- OpenShift AI cluster with TrustyAI Operator installed
- A model deployed via KServe with `security.opendatahub.io/enable-auth: "true"`
- `oc` CLI logged in to the cluster
- Hugging Face token for gated model access

## Step 1: Set Your Configuration

Configuration is loaded from `../.env`. Copy `sample.env` to `.env` and update values before running:

In [8]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

NAMESPACE = os.getenv("NAMESPACE", "hyo-project")
HF_TOKEN = os.getenv("HF_TOKEN", "hf_xxxxx")
HF_TOKEN_SECRET = os.getenv("HF_TOKEN_SECRET", "hf-token")

print(f"Namespace: {NAMESPACE}")
print(f"HF Token: {HF_TOKEN[:8]}...")
print(f"HF Token Secret: {HF_TOKEN_SECRET}")

Namespace: rhoai-models
HF Token: hf_wLpqt...
HF Token Secret: hf-token


## Step 2: Verify Cluster Access

Make sure you are logged in to the OpenShift cluster. If not, run:

```bash
oc login --server=<your-cluster-api-url>
```

In [9]:
!oc whoami
!oc project {NAMESPACE}

kube:admin
Already on project "rhoai-models" on server "https://api.openshift-cluster.sandbox1785.opentlc.com:6443".


## Step 3: Identify Your Model

Find the InferenceService name — this is also the `served_model_name` used by vLLM:

In [10]:
!oc get inferenceservice -n {NAMESPACE}

No resources found in rhoai-models namespace.


## Step 4: Create RBAC Permissions

The LMEvalJob Pod uses the `default` ServiceAccount. When the InferenceService has OAuth enabled, the SA needs permission to `get` InferenceServices for the OAuth proxy to validate access.

In [11]:
!oc apply -f samples/role.yaml -n {NAMESPACE}
!oc apply -f samples/rolebinding.yaml -n {NAMESPACE}

role.rbac.authorization.k8s.io/inferenceservice-reader unchanged
rolebinding.rbac.authorization.k8s.io/lmeval-inferenceservice-access unchanged


## Step 5: Create Hugging Face Token Secret

Required for downloading gated tokenizers (e.g., `google/gemma-2b`).

> **Note:** If the namespace was created via the RHOAI Data Science Project UI, a `hf-token` secret may already exist with a `token` key. LMEvalJob expects an `HF_TOKEN` key. The cell below uses `oc patch` to add/overwrite the `HF_TOKEN` key without disturbing other keys, so it is safe to re-run at any time.

In [12]:
# Create the secret if it doesn't exist, then patch to ensure HF_TOKEN key is present.
# RHOAI UI may overwrite secrets created via 'oc apply', so we use 'oc patch' to
# guarantee the HF_TOKEN key survives regardless of execution order.
!oc create secret generic {HF_TOKEN_SECRET} -n {NAMESPACE} --dry-run=client -o yaml | oc apply -f - 2>/dev/null; \
  oc patch secret {HF_TOKEN_SECRET} -n {NAMESPACE} --type merge \
    -p '{{"stringData":{{"HF_TOKEN":"{HF_TOKEN}","hf-token":"{HF_TOKEN}"}}}}'

# Verify
!echo "Keys in secret:" && oc get secret {HF_TOKEN_SECRET} -n {NAMESPACE} -o jsonpath='{{.data}}' | python3 -c \
    "import sys,json; d=json.load(sys.stdin); [print(f'  {{k}}: {{\"set\" if v else \"EMPTY\"}}') for k,v in d.items()]"

secret/hf-token configured
secret/hf-token patched
Keys in secret:
  HF_TOKEN: set
  hf-token: set


## Step 6: Create Service Account Token Secret

This long-lived SA token is injected as `OPENAI_API_KEY` to authenticate with the OAuth-protected model endpoint:

In [13]:
!oc apply -f samples/sa-token-secret.yaml -n {NAMESPACE}

secret/lmeval-sa-token unchanged


## Step 7: Verify Permissions

In [14]:
!oc auth can-i get inferenceservices.serving.kserve.io \
    -n {NAMESPACE} \
    --as=system:serviceaccount:{NAMESPACE}:default

yes


Expected output: `yes`

## Done!

You're now ready to run evaluations. Proceed to:
- **0_setup/2_eval_hub_setup.ipynb** to configure EvalHub SDK with MLflow experiment tracking
- **1_eval_hub_guidellm_benchmark/1_guidellm_benchmark.ipynb** for inference performance benchmarks
- **2_eval_hub_kmcq_benchmark/1_kmcq_benchmark.ipynb** for Korean MCQ benchmark evaluation
- **2_eval_hub_kmcq_benchmark/2_summarize_results.ipynb** for summarizing results and generating reports
- **3_eval_hub_unified_benchmark/1_unified_benchmark.ipynb** for unified accuracy + performance evaluation